# Entity Disambiguation Diagnostic

After promoting the normalization rules that look safe here, rerun `31_entity_disambiguation.ipynb` and compare the episode and person coverage tables again. The goal is to keep tightening the obvious-match gap without changing the deterministic ordering or the event-sourced replay behavior.

## Optional Step 0: Rebuild Step 311 Outputs (Clean State)

Run this cell when code changed and projections/events need to be regenerated before diagnostics.

In [ ]:
import shutil
import sys
from pathlib import Path

repo_root = Path.cwd()
for parent in [repo_root] + list(repo_root.parents):
    if (parent / "speakermining" / "src" / "process").exists():
        repo_root = parent
        break

src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from process.entity_disambiguation import Step311Orchestrator
from process.entity_disambiguation.config import PHASE_DIR

REBUILD_CLEAN = True

if REBUILD_CLEAN:
    for p in [PHASE_DIR / "events", PHASE_DIR / "checkpoints", PHASE_DIR / "projections"]:
        if p.exists():
            shutil.rmtree(p)
    db = PHASE_DIR / "handler_progress.db"
    if db.exists():
        db.unlink()

orchestrator = Step311Orchestrator()
projections = orchestrator.run()
print("Rebuild finished")
for key, df in projections.items():
    print(f"{key}: rows={len(df)} cols={len(df.columns)}")

## Step 1: Load Projections and Source Inputs

Load aligned outputs from Step 311 and raw Wikidata projection JSON files used during alignment.

In [ ]:
from pathlib import Path
import json
import pandas as pd

repo_root = Path.cwd()
for parent in [repo_root] + list(repo_root.parents):
    if (parent / "speakermining" / "src" / "process").exists():
        repo_root = parent
        break

phase31_dir = repo_root / "data" / "31_entity_disambiguation" / "projections"
wd_proj_dir = repo_root / "data" / "20_candidate_generation" / "wikidata" / "projections"

aligned_files = {
    "broadcasting_programs": phase31_dir / "aligned_broadcasting_programs.csv",
    "series": phase31_dir / "aligned_series.csv",
    "episodes": phase31_dir / "aligned_episodes.csv",
    "persons": phase31_dir / "aligned_persons.csv",
    "topics": phase31_dir / "aligned_topics.csv",
    "roles": phase31_dir / "aligned_roles.csv",
    "organizations": phase31_dir / "aligned_organizations.csv",
}

aligned = {}
for key, path in aligned_files.items():
    aligned[key] = pd.read_csv(path, dtype=str, keep_default_na=False, low_memory=False) if path.exists() else pd.DataFrame()

wd_files = {
    "broadcasting_programs": wd_proj_dir / "broadcasting_programs.json",
    "series": wd_proj_dir / "series.json",
    "episodes": wd_proj_dir / "episodes.json",
    "persons": wd_proj_dir / "persons.json",
    "topics": wd_proj_dir / "topics.json",
    "roles": wd_proj_dir / "roles.json",
    "organizations": wd_proj_dir / "organizations.json",
}

wd_raw = {}
for key, path in wd_files.items():
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
            wd_raw[key] = payload if isinstance(payload, dict) else {}
    else:
        wd_raw[key] = {}

print(f"Repo root: {repo_root}")
print(f"Loaded aligned frames: {', '.join(sorted(aligned.keys()))}")
print(f"Loaded Wikidata raw sets: {', '.join(sorted(wd_raw.keys()))}")

## Step 2: Audit Wikidata Property Coverage in Aligned Outputs

This check compares which Wikidata claim properties exist in the source JSON vs. which are currently projected into aligned CSV columns.

In [ ]:
def property_frequency(entity_dict: dict) -> pd.DataFrame:
    counts = {}
    for entity in entity_dict.values():
        claims = (entity or {}).get("claims", {}) or {}
        for pid, statements in claims.items():
            if not isinstance(statements, list):
                continue
            counts[pid] = counts.get(pid, 0) + len(statements)
    rows = [{"property": pid, "statement_count": n} for pid, n in counts.items()]
    if not rows:
        return pd.DataFrame(columns=["property", "statement_count"])
    return pd.DataFrame(rows).sort_values("statement_count", ascending=False).reset_index(drop=True)

# What columns do we currently expose from Wikidata in aligned CSVs?
current_wd_columns = {
    "broadcasting_programs": sorted([c for c in aligned["broadcasting_programs"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
    "series": sorted([c for c in aligned["series"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
    "episodes": sorted([c for c in aligned["episodes"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
    "persons": sorted([c for c in aligned["persons"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
    "topics": sorted([c for c in aligned["topics"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
    "roles": sorted([c for c in aligned["roles"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
    "organizations": sorted([c for c in aligned["organizations"].columns if c.startswith("wikidata_") or "wikidata" in c.lower() or c.endswith("_id") or c.endswith("_label")]),
}

coverage_rows = []
for core_class, entities in wd_raw.items():
    freq = property_frequency(entities)
    top_props = freq.head(15)["property"].tolist() if not freq.empty else []
    wd_cols = current_wd_columns.get(core_class, [])
    has_full_payload_cols = (
        "wikidata_claim_properties" in wd_cols
        and "wikidata_property_counts_json" in wd_cols
        and "wikidata_claim_property_count" in wd_cols
    )
    coverage_rows.append(
        {
            "core_class": core_class,
            "entity_count": len(entities),
            "distinct_wikidata_properties": int(freq["property"].nunique()) if not freq.empty else 0,
            "top_15_properties": ", ".join(top_props),
            "current_wd_related_columns": ", ".join(wd_cols),
            "current_wd_column_count": len(wd_cols),
            "has_full_property_payload_columns": has_full_payload_cols,
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values("core_class").reset_index(drop=True)
coverage_df

## Step 3: Detect Series Misclassified as Broadcasting Programs

Detect Wikidata entities in broadcasting_programs.json where P31 contains Q3464665 (television series season), and check whether Step 311 routes them into aligned_series.csv.

In [ ]:
def entity_has_p31_qid(entity: dict, qid: str) -> bool:
    claims = (entity or {}).get("claims", {}) or {}
    p31_statements = claims.get("P31", [])
    if not isinstance(p31_statements, list):
        return False
    for stmt in p31_statements:
        mainsnak = (stmt or {}).get("mainsnak", {}) or {}
        datavalue = (mainsnak or {}).get("datavalue", {}) or {}
        value = (datavalue or {}).get("value", {}) or {}
        if isinstance(value, dict) and value.get("id") == qid:
            return True
    return False

def entity_label(entity: dict) -> str:
    labels = (entity or {}).get("labels", {}) or {}
    return ((labels.get("de") or {}).get("value") or (labels.get("en") or {}).get("value") or "").strip()

series_p31_qid = "Q3464665"  # television series season
wd_broadcasting = wd_raw.get("broadcasting_programs", {})

season_like_in_broadcasting = []
for qid, entity in wd_broadcasting.items():
    if entity_has_p31_qid(entity, series_p31_qid):
        season_like_in_broadcasting.append(
            {
                "qid": qid,
                "label": entity_label(entity),
                "has_P179": "P179" in ((entity or {}).get("claims", {}) or {}),
                "has_P580": "P580" in ((entity or {}).get("claims", {}) or {}),
            }
        )

season_like_df = pd.DataFrame(season_like_in_broadcasting)

aligned_series = aligned.get("series", pd.DataFrame())
aligned_broadcasting = aligned.get("broadcasting_programs", pd.DataFrame())

series_qids = set(aligned_series.get("series_id", pd.Series(dtype=str)).dropna().astype(str))
broadcasting_qids = set(aligned_broadcasting.get("source_wikidata_value", pd.Series(dtype=str)).dropna().astype(str))

if not season_like_df.empty:
    season_like_df["present_in_aligned_series"] = season_like_df["qid"].isin(series_qids)
    season_like_df["present_in_aligned_broadcasting"] = season_like_df["qid"].isin(broadcasting_qids)

season_routing_summary = pd.DataFrame(
    [
        {
            "total_q3464665_in_wd_broadcasting": len(season_like_df),
            "found_in_aligned_series": int(season_like_df["present_in_aligned_series"].sum()) if not season_like_df.empty else 0,
            "found_in_aligned_broadcasting": int(season_like_df["present_in_aligned_broadcasting"].sum()) if not season_like_df.empty else 0,
            "missing_from_aligned_series": int((~season_like_df["present_in_aligned_series"]).sum()) if not season_like_df.empty else 0,
        }
    ]
)

season_routing_summary

## Step 4: Redesign Gap Summary (Actionable)

Generate a compact summary showing where Step 311 output does not yet meet redesign expectations.

In [ ]:
gap_rows = []

# Gap A: Property projection coverage
for _, row in coverage_df.iterrows():
    if bool(row["has_full_property_payload_columns"]):
        severity = "low"
        observed = (
            f"full payload columns present; {row['distinct_wikidata_properties']} WD properties represented via "
            "wikidata_claim_properties + wikidata_property_counts_json"
        )
    else:
        severity = "high"
        observed = (
            f"missing full payload columns; {row['distinct_wikidata_properties']} distinct WD properties "
            f"vs {row['current_wd_column_count']} WD-related columns"
        )

    gap_rows.append(
        {
            "gap_type": "wikidata_property_coverage",
            "core_class": row["core_class"],
            "severity": severity,
            "observed": observed,
            "recommended_fix": "Ensure all aligned projections include wikidata_claim_properties, wikidata_claim_property_count, wikidata_property_counts_json",
        }
    )

# Gap B: Season-like entities not routed to aligned_series
if not season_like_df.empty:
    missing_in_series = season_like_df[~season_like_df["present_in_aligned_series"]]
    gap_rows.append(
        {
            "gap_type": "series_misclassification",
            "core_class": "series",
            "severity": "high" if len(missing_in_series) > 0 else "low",
            "observed": f"{len(missing_in_series)} of {len(season_like_df)} Q3464665 entities from broadcasting_programs not found in aligned_series",
            "recommended_fix": "Route broadcasting_program entities with P31=Q3464665 into series projection as additional seeds",
        }
    )

gap_df = pd.DataFrame(gap_rows)
severity_rank = {"high": 0, "medium": 1, "low": 2}
gap_df["severity_rank"] = gap_df["severity"].map(severity_rank).fillna(99)
gap_df = gap_df.sort_values(["severity_rank", "gap_type", "core_class"], ascending=[True, True, True]).drop(columns=["severity_rank"]).reset_index(drop=True)
gap_df

In [ ]:
summary = {
    "classes_with_full_property_payload": int(coverage_df["has_full_property_payload_columns"].sum()),
    "total_classes": int(len(coverage_df)),
    "remaining_high_gaps": int((gap_df["severity"] == "high").sum()),
}
print(summary)
coverage_df[["core_class", "distinct_wikidata_properties", "current_wd_column_count", "has_full_property_payload_columns"]]

In [ ]:
# Compact metrics for documentation handover
metrics = {
    "classes_with_full_property_payload": int(coverage_df["has_full_property_payload_columns"].sum()),
    "total_classes": int(len(coverage_df)),
    "remaining_high_gaps": int((gap_df["severity"] == "high").sum()),
    "q3464665_total_in_wd_broadcasting": int(season_routing_summary.loc[0, "total_q3464665_in_wd_broadcasting"]),
    "q3464665_found_in_aligned_series": int(season_routing_summary.loc[0, "found_in_aligned_series"]),
    "q3464665_missing_from_aligned_series": int(season_routing_summary.loc[0, "missing_from_aligned_series"]),
}
metrics